In [1]:
import torch

In [2]:
first_half = []
p = 4
denom_base = torch.tensor(10000)

for i in range(0, 128//2):
    sin_component = torch.sin(p / (denom_base**(2*i/128)))
    cos_component = torch.cos(p / (denom_base**(2*i/128)))
    
    first_half.append(sin_component)
    first_half.append(cos_component)

first_half = torch.tensor(first_half)

In [3]:
second_half = []
p = 4
denom_base = torch.tensor(10000)

for i in range(0, 128//2):
    sin_component = torch.sin(p / (denom_base**(2*i/128)))
    cos_component = torch.cos(p / (denom_base**(2*i/128)))
    
    second_half.append(sin_component)
    second_half.append(cos_component)

second_half = torch.tensor(second_half)

In [4]:
torch.cat((first_half, second_half)).shape

torch.Size([256])

In [5]:
randn = torch.randn(256, 7, 7)

In [6]:
randn = randn.permute(1, 2, 0)
randn.shape, randn[0][0][:10], randn[0][0][-10:]

(torch.Size([7, 7, 256]),
 tensor([-0.3861, -0.3689, -0.3025,  0.0104, -0.4909, -0.8064, -0.8024, -0.2852,
          0.6579,  0.5449]),
 tensor([ 0.5124,  0.2920, -1.3584, -0.9135, -0.6218, -0.6584, -1.4835, -0.3930,
         -2.0784,  0.8475]))

In [12]:
from torch import sin, cos

for pos_y in range(randn.shape[0]): # rows
    for pos_x in range(randn.shape[1]): # columns
        y_half, x_half = torch.empty(128), torch.empty(128)
        
        for i in range(0, 256 // 4):
            scale = denom_base**(2*i/128)
            argument_y = pos_y / scale
            argument_x = pos_x / scale
            
            y_half[2*i] = sin(argument_y)
            y_half[2*i+1] = cos(argument_y)
            
            x_half[2*i] = sin(argument_x)
            x_half[2*i+1] = cos(argument_x)
        
        pos_embedding = torch.cat((x_half, y_half), dim=-1)
        
        print(pos_embedding[:10], pos_embedding[-10:])

tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.]) tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([0.8415, 0.5403, 0.7617, 0.6479, 0.6816, 0.7318, 0.6047, 0.7965, 0.5332,
        0.8460]) tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([ 0.9093, -0.4161,  0.9870, -0.1604,  0.9975,  0.0709,  0.9632,  0.2687,
         0.9021,  0.4315]) tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([ 0.1411, -0.9900,  0.5173, -0.8558,  0.7783, -0.6279,  0.9296, -0.3685,
         0.9933, -0.1160]) tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([-0.7568, -0.6536, -0.3167, -0.9485,  0.1415, -0.9899,  0.5176, -0.8556,
         0.7785, -0.6277]) tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([-0.9589,  0.2837, -0.9277, -0.3733, -0.5711, -0.8209, -0.1051, -0.9945,
         0.3239, -0.9461]) tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])
tensor([-0.2794,  0.9602, -0.8854,  0.4648, -0.9774, -0.2114, -0.6851, -0.7285,
        -0.2304, -0.9731]) tensor([0., 1., 0., 1., 0

In [45]:
H = 7
W = 7
D = 256

pos_encodings = torch.empty(H, W, D)

for pos_y in range(H):  # rows
    for pos_x in range(W):  # columns
        # compute y's half and x's half of positional encoding
        for i in range(0, D // 4):
            # sinusoidal positional encoding from "Attention Is All You
            # Need" paper
            scale = torch.tensor(10_000 ** (2 * i / (D / 2)))
            argument_y = pos_y / scale
            argument_x = pos_x / scale

            if pos_y == 0 and pos_x == 1 and i == 59:
                print(argument_x.item())
                print(sin(argument_x).item())
            
            pos_encodings[pos_y][pos_x][2*i] = sin(argument_y)
            pos_encodings[pos_y][pos_x][2*i+1] = cos(argument_y)

            pos_encodings[pos_y][pos_x][(2*i)+128] = sin(argument_x)
            pos_encodings[pos_y][pos_x][(2*i+1)+128] = cos(argument_x)

0.00020535249495878816
0.00020535249495878816


In [38]:
pos_encodings[0][1][246:]

tensor([2.0535e-04, 1.0000e+00, 1.7783e-04, 1.0000e+00, 1.5399e-04, 1.0000e+00,
        1.3335e-04, 1.0000e+00, 1.1548e-04, 1.0000e+00])

In [44]:
sin(torch.tensor(0.00020535249495878816))

tensor(0.0002)

In [47]:
pos_encodings = pos_encodings.reshape(H*W, D)
pos_encodings.shape

torch.Size([49, 256])

In [57]:
pos_encodings.unsqueeze(0).unsqueeze(0).shape

torch.Size([1, 1, 49, 256])